In [ ]:
import os
import math
from utils import *

import pennylane as qml
from pennylane import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 42

# Generating the dataset

We will generate a synthetic dataset, generated by solving the Lorentz system using the Euler method. The Lorenz equations are defined as:

$$
\dot{x} = \sigma (y-x)
$$
$$
\dot{y} = -y -zx + \rho x
$$
$$
\dot{z} = -\beta z +xy
$$

where $(x, y, z)$ are the variables and $(\sigma , \rho , \beta)$ are parameters. 

In [ ]:
npoints = 1000
h = 0.01
params = [10, 28, 8 / 3]
init_point = [0, -0.01, 9]

dataset = generate_lorenz(
    npoints, 
    h,
    params, 
    init_point
)

# Trin test split
dataset_train, dataset_val = train_test_split(
    dataset,
    test_size=0.25,
    shuffle=False
)
time = np.arange(npoints) * h

time_tr = time[:len(dataset_train)]
time_val = time[len(dataset_train):]

# Scale the data to the range [0, 1]
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(dataset_train)
val_scaled = scaler.transform(dataset_val)
val_scaled = np.clip(val_scaled, 0, 1)

plot_lorenz(train_scaled, val_scaled, time_tr, time_val)

# iQTransformer

The iQTransformer architecture follows the same pipeline as the iTransformer, with the exception of the self-attention layer in the transformers blocks, which is replaced by a Quantum Self-Attention Neural Network (QSANN). This mechanism replaces the linear projection of queries, keys, and values with VQCs. Each Quantum Self-Attention Layer (QSAL) proceeds in three stages.

First, each token $h^{l-1}\in \mathbb{R}^D$ is encoded into an $n$-qubit quantum state:

$$
|\psi \rangle = U_{enc}(h_c^{l-1})H^{\otimes n}|0\rangle ^{\otimes n}, \quad c=1,...,C,
$$

where $U_{enc}$ is a quantum data-encoding ansatz.

Second, three different variational circuits $U_q(\theta_q)$, $U_k(\theta_k)$ and $U_v(\theta_v)$ representing the query, key and value are applied. The query and the key are represented as Pauli-Z expectation value on a single qubit, while the value is represented by a $D$-dimensional vector of expected values:

$$
q_c = \langle Z_q \rangle_c = \langle \psi_c | U_q^{\dagger} (\theta_q) Z_0 U_q (\theta_q)|\psi_c \rangle,
$$

$$
k_c = \langle Z_k \rangle_c = \langle \psi_c | U_k^{\dagger} (\theta_k) Z_0 U_k (\theta_k)|\psi_c \rangle,
$$

$$
v_c = (\langle P_1 \rangle_c,...,\langle P_D \rangle_c)^T, \quad \langle P_j \rangle_c = \langle \psi_c | U_v^{\dagger} (\theta_v) P_j U_v (\theta_v)|\psi_v \rangle.
$$

Finally, the attention coefficients are computed by Gaussian projection of the query and key measurements:

$$
\alpha_{c,c'} = exp(-(\langle Z_q \rangle_c - \langle Z_k \rangle_{c'})^2), \quad \tilde{\alpha}_{c,c'}=\frac{\alpha_{c,c'}}{\sum_{m=1}^C\alpha_{c,m}}
$$

Amd the layer output is updated as:

$$
h_c^l = h_c^{l-1} + \sum_{c'=1}^C \tilde{\alpha}_{c,c'} v_{c'}
$$

The same ansatz is used for all $U_{enc}$, $U_{q}$, $U_{k}$, and $U_{v}$, consisting of layers of $R_{x}$ and $R_{y}$ rotations, followed by $p$ repetitions of an entangling block. This block is composed of circular entanglement with CNOT gates and a layer of $R_{y}$ rotations. Consequently, both the encoding and the ansatz have the same dimension, which is given by

$$
D = n(p+2)
$$

Regarding the value vectors, the operators $\{P_j\}$ are taken from a fixed set of Pauli observables, and their choice determines the output dimension. For the simplest case, one can select local measurements such as $\{X_i,Y_i,Z_i\}$ on each qubit $i=1,..,n$, yielding $D=3n$ observables in total. For deeper encodings $(p>1)$, additional two-qubit observables could be included.

In [ ]:
# Params
batch_size = 128 
n_chanels = 3
window_size = 5
batch_size = 128
n_trfm_blocks = 2 
embding_dim = 9
ff_dim = 12
nqubits = 3
enc_depth = 1
variational_depth = 3
weigths = {"theta": (nqubits * (variational_depth + 2),)}

# Training hyperparameters
epochs = 50
lr = 5e-4

In [ ]:
dev = qml.device("lightning.qubit", wires=nqubits)

def QSA_query_key(inputs, theta):
    for i in range(nqubits):
        qml.H(wires=i)
    QSANN_embedding(nqubits, inputs, enc_depth)
    QSANN_VQC(nqubits, theta, variational_depth)

    return qml.expval(qml.Z(0))

def QSA_value(inputs, theta):
    for i in range(nqubits):
        qml.H(wires=i)
    QSANN_embedding(nqubits, inputs, enc_depth)
    QSANN_VQC(nqubits, theta, variational_depth)
    value = []
    for i in range(nqubits):
        value.append(qml.expval(qml.X(i)))
        value.append(qml.expval(qml.Y(i)))
        value.append(qml.expval(qml.Z(i)))
    # Returns 3n observables
    return value

QSA_query_key_node = qml.QNode(
    QSA_query_key,
    device=dev,
    interface="torch",
    diff_method="adjoint"
)

QSA_value_node = qml.QNode(
    QSA_value,
    device=dev,
    interface="torch",
    diff_method="adjoint"
)

class QuantumSelfAttentionLayer(nn.Module):
    def __init__(self,
                eps=1e-5
        ):
        super().__init__()

        self.eps = eps

        self.q_layer = qml.qnn.TorchLayer(QSA_query_key_node, weigths)
        self.k_layer = qml.qnn.TorchLayer(QSA_query_key_node, weigths)
        self.v_layer = qml.qnn.TorchLayer(QSA_value_node, weigths)


    def forward(self, x):
        # x.shape = (batch, chanels, embding_dim) 
        Q, K, V = [], [], []
        
        n_channels = x.shape[1]

        for c in range(n_channels):
            token = x [:, c, :]

            Q.append(self.q_layer(token)) # (B,)
            K.append(self.k_layer(token)) # (B,)
            V.append(self.v_layer(token)) # (B, 3 * nqubits)
        
        Q = torch.stack(Q, dim=1)  # (B, C)
        K = torch.stack(K, dim=1)  # (B, C)
        V = torch.stack(V, dim=1)  # (B, C, D)

        # Broadcasting: (B, C) → (B, C, 1) - (B, C) → (B, 1, C) = (B, C, C) 
        differences = torch.exp(-(Q.unsqueeze(2) - K.unsqueeze(1)) ** 2)

        attention_weights = differences / (
            differences.sum(dim=-1, keepdim=True) + self.eps
        )
        update = attention_weights @ V

        return update


class QuantumTransformerBlock(nn.Module):
    def __init__(self,
                embding_dim,
                ff_dim,
                eps=1e-5
        ):
        super().__init__()

        self.eps = eps

        self.QSA_layer = QuantumSelfAttentionLayer()

        self.feedforward = nn.Sequential(
            nn.Linear(embding_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embding_dim)
        )

    def normalize(self, x):
        # (batch, chanels, embding_dim) 
        mean = x.mean(dim=1, keepdim=True)
        var = x.var(dim=1, keepdim=True, unbiased=False)
        std = torch.sqrt(var + self.eps)

        x_normalized = (x - mean) / std

        return x_normalized

    def forward(self, x):
        x = x + self.QSA_layer(self.normalize(x))
        x = x + self.feedforward(self.normalize(x))
        return x


class iQTransformer_Model(nn.Module):
    def __init__(self, 
                window_size, 
                horizon, 
                n_trfm_blocks,
                embding_dim,
                ff_dim,
                eps=1e-5
        ):
        super().__init__()

        self.eps = eps

        self.token_embedding = nn.Linear(
            window_size, 
            embding_dim
        )

        self.trfm_blocks = nn.ModuleList([
            QuantumTransformerBlock(embding_dim, ff_dim)
            for _ in range(n_trfm_blocks)
        ])

        self.output_projection = nn.Linear(
            embding_dim, 
            horizon
        )

    def normalize(self, x):
        mean = x.mean(dim=2, keepdim=True)
        var = x.var(dim=2, keepdim=True, unbiased=False)
        std = torch.sqrt(var + self.eps)

        x_normalized = (x - mean) / std

        return x_normalized, mean, std

    def denormalize(self, x, mean, std):
        return x * std + mean
    

    def forward(self, x):
        # x.shape = (batch, window_size, chanels)

        x = x.transpose(1, 2) # (batch, chanels, window_size)

        x, mean, std = self.normalize(x)

        x = self.token_embedding(x) # (batch, chanels, embding_dim) 

        for block in self.trfm_blocks:
            x = block(x)

        x = self.output_projection(x) # (batch, chanels, horizon) 

        x = self.denormalize(x, mean, std)

        x = x.transpose(1, 2) # (batch, horizon, chanels) 

        return x

# Short-term forecasting

In [ ]:
# Create the sequences
horizon_short = 1

x_tr_short, hor_tr_short = create_sequences(train_scaled, window_size, horizon_short)
x_val_short, hor_val_short = create_sequences(val_scaled, window_size, horizon_short)

x_tr_short = torch.from_numpy(x_tr_short).double()
hor_tr_short = torch.from_numpy(hor_tr_short).double()

x_val_short = torch.from_numpy(x_val_short).double()
hor_val_short = torch.from_numpy(hor_val_short).double()

tr_dataset_short = TensorDataset(x_tr_short, hor_tr_short)
tr_loader_short = DataLoader(tr_dataset_short, batch_size=batch_size, shuffle=False)

In [ ]:
reset_seeds(SEED)

# Create the model
model_iQTransformer_short = iQTransformer_Model(
    window_size, 
    horizon_short, 
    n_trfm_blocks,
    embding_dim,
    ff_dim,
).double()

# Define optimizer and loss function
opt_short = torch.optim.Adam(params=model_iQTransformer_short.parameters(), lr=lr)
loss_fn = nn.MSELoss()

# Save the best model
best_state_short = None
best_val_loss_short = float("inf")

history_short = {"Loss": [], "Val loss": []}

print(30 * "#")
print("Starting training")
print(30 * "#")

for epoch in range(epochs):

    #Training
    model_iQTransformer_short.train()
    epoch_loss = 0
    for xb, yb in tr_loader_short:
        opt_short.zero_grad()
        pred = model_iQTransformer_short(xb)
        pred = pred.reshape(pred.shape[0], -1)
        target = yb.reshape(yb.shape[0], -1)
        loss = loss_fn(pred, target)    
        loss.backward()
        opt_short.step()
        epoch_loss += loss.item()
    
    # Store average loss for this epoch
    epoch_loss /= len(tr_loader_short)
    history_short["Loss"].append(epoch_loss)

    # Evaluation
    model_iQTransformer_short.eval()
    with torch.no_grad():
        val_pred = model_iQTransformer_short(x_val_short)
        val_pred = val_pred.reshape(val_pred.shape[0], -1)
        val_target = hor_val_short.reshape(hor_val_short.shape[0], -1)
        val_loss = loss_fn(val_pred, val_target).item()

    # Store val loss
    history_short["Val loss"].append(val_loss)

    # Save best model
    if val_loss < best_val_loss_short:
        best_val_loss_short = val_loss
        best_state_short = {
            name: params.detach().clone()
            for name, params in model_iQTransformer_short.state_dict().items()
        }

    # Print training progress every 5 epochs
    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"Epoch: {epoch + 1} | Loss: {epoch_loss:.4f} | Validation loss: {val_loss:.4f}")


if best_state_short is not None:
    model_iQTransformer_short.load_state_dict(best_state_short)

In [ ]:
# Save the model
os.makedirs("../models_states", exist_ok=True)

checkpoint = {
    "iQTranformerShort": model_iQTransformer_short.state_dict(),
    "history": history_short
}
torch.save(checkpoint, "../models_states/iQTransformer_short.pt")

In [ ]:
# # Load the model
# checkpoint = torch.load(
#     "../models_states/iQTransformer_short.pt",
#     map_location="cpu"
# )

# history_short = checkpoint["history"]

# model_iQTransformer_short = iQTransformer_Model(
#     window_size, 
#     horizon_short, 
#     n_trfm_blocks,
#     embding_dim,
#     ff_dim,
# ).double()
# model_iQTransformer_short.load_state_dict(checkpoint["iQTranformerShort"])

In [ ]:
plot_loss(history_short, "iQTransformer - Short term")

In [ ]:
# Evaluation mode
model_iQTransformer_short.eval()

with torch.no_grad():
    pred_short = model_iQTransformer_short(x_val_short)

pred_3d_short = pred_short.reshape(pred_short.shape[0], 1, 3)
target_3d_short = hor_val_short

# =====================
# RMSE
# =====================

squared_error_short = (pred_3d_short - target_3d_short) ** 2
mean_rmse_list_short = torch.sqrt(squared_error_short.mean(dim=(1, 2)))
mean_rmse_short = torch.sqrt(squared_error_short.mean())

# =====================
# MAE
# =====================

abs_error_short = torch.abs(pred_3d_short - target_3d_short)
mean_mae_list_short = abs_error_short.mean(dim=(1, 2))
mean_mae_short = abs_error_short.mean()

# =====================
# MAPE
# =====================

eps = 1e-8

percentage_error_short = torch.abs((target_3d_short - pred_3d_short) / (target_3d_short + eps))
mean_mape_list_short = percentage_error_short.mean(dim=(1, 2))
mean_mape_short = percentage_error_short.mean()

print(f"Global RMSE: {mean_rmse_short:.4f}")
print(f"Global MAE: {mean_mae_short:.4f}")
print(f"Global MAPE: {mean_mape_short:.4f}")

In [ ]:
# Save the results to a CSV file
save_result_csv(
    "../results/lorenz_metrics.csv",
    {
        "forecasting": "short-term",
        "model": "iQTransformer",
        "MAPE": mean_mape_short,
        "MAE": mean_mae_short,
        "RMSE": mean_rmse_short,
    },
)

In [ ]:
pred_x_short = pred_3d_short[:, :, 0]
pred_y_short = pred_3d_short[:, :, 1]
pred_z_short = pred_3d_short[:, :, 2]

plot_preds_and_error(
    pred_x_short,
    pred_y_short,
    pred_z_short,
    mean_mae_list_short,
    time_val,
    window_size,
    val_scaled,
    error="MAE",
    title="iQTransformer short-term"
)

# Long-term forecasting

In [ ]:
# Create the sequences
horizon_long = 5

x_tr_long, hor_tr_long = create_sequences(train_scaled, window_size, horizon_long)
x_val_long, hor_val_long = create_sequences(val_scaled, window_size, horizon_long)

x_tr_long = torch.from_numpy(x_tr_long).double()
hor_tr_long = torch.from_numpy(hor_tr_long).double()

x_val_long = torch.from_numpy(x_val_long).double()
hor_val_long = torch.from_numpy(hor_val_long).double()

tr_dataset_long = TensorDataset(x_tr_long, hor_tr_long)
tr_loader_long = DataLoader(tr_dataset_long, batch_size=batch_size, shuffle=False)

In [ ]:
reset_seeds(SEED)

# Create the model
model_iQTransformer_long = iQTransformer_Model(
    window_size, 
    horizon_long, 
    n_trfm_blocks,
    embding_dim,
    ff_dim,
).double()

# Define optimizer and loss function
opt_long = torch.optim.Adam(params=model_iQTransformer_long.parameters(), lr=lr)
loss_fn = nn.MSELoss()

# Save the best model
best_state_long = None
best_val_loss_long = float("inf")

history_long = {"Loss": [], "Val loss": []}

print(30 * "#")
print("Starting training")
print(30 * "#")

for epoch in range(epochs):

    #Training
    model_iQTransformer_long.train()
    epoch_loss = 0
    for xb, yb in tr_loader_long:
        opt_long.zero_grad()
        pred = model_iQTransformer_long(xb)
        pred = pred.reshape(pred.shape[0], -1)
        target = yb.reshape(yb.shape[0], -1)
        loss = loss_fn(pred, target)    
        loss.backward()
        opt_long.step()
        epoch_loss += loss.item()
    
    # Store average loss for this epoch
    epoch_loss /= len(tr_loader_long)
    history_long["Loss"].append(epoch_loss)

    # Evaluation
    model_iQTransformer_long.eval()
    with torch.no_grad():
        val_pred = model_iQTransformer_long(x_val_long)
        val_pred = val_pred.reshape(val_pred.shape[0], -1)
        val_target = hor_val_long.reshape(hor_val_long.shape[0], -1)
        val_loss = loss_fn(val_pred, val_target).item()

    # Store val loss
    history_long["Val loss"].append(val_loss)

    # Save best model
    if val_loss < best_val_loss_long:
        best_val_loss_long = val_loss
        best_state_long = {
            name: params.detach().clone()
            for name, params in model_iQTransformer_long.state_dict().items()
        }

    # Print training progress every 5 epochs
    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"Epoch: {epoch + 1} | Loss: {epoch_loss:.4f} | Validation loss: {val_loss:.4f}")


if best_state_long is not None:
    model_iQTransformer_long.load_state_dict(best_state_long)

In [ ]:
# Save the model
os.makedirs("../models_states", exist_ok=True)

checkpoint = {
    "iQTrasnformerLong": model_iQTransformer_long.state_dict(),
    "history": history_long
}
torch.save(checkpoint, "../models_states/iQTransformer_long.pt")

In [ ]:
# # Load the model
# checkpoint = torch.load(
#     "../models_states/iQTransformer_long.pt",
#     map_location="cpu"
# )

# history_long = checkpoint["history"]

# model_iQTransformer_long = iQTransformer_Model(
#     window_size, 
#     horizon_long, 
#     n_trfm_blocks,
#     embding_dim,
#     ff_dim,
# ).double()
# model_iQTransformer_long.load_state_dict(checkpoint["iQTrasnformerLong"])

In [ ]:
plot_loss(history_long, "iQTransformer - Long term")

In [ ]:
# Evaluation mode
model_iQTransformer_long.eval()

with torch.no_grad():
    pred_long = model_iQTransformer_long(x_val_long)

pred_3d_long = pred_long.reshape(pred_long.shape[0], 5, 3)
target_3d_long = hor_val_long

# =====================
# RMSE
# =====================

squared_error_long = (pred_3d_long - target_3d_long) ** 2
mean_rmse_list_long = torch.sqrt(squared_error_long.mean(dim=(1, 2)))
mean_rmse_long = torch.sqrt(squared_error_long.mean())

# =====================
# MAE
# =====================

abs_error_long = torch.abs(pred_3d_long - target_3d_long)
mean_mae_list_long = abs_error_long.mean(dim=(1, 2))
mean_mae_long = abs_error_long.mean()

# =====================
# MAPE
# =====================

eps = 1e-8

percentage_error_long = torch.abs((target_3d_long - pred_3d_long) / (target_3d_long + eps))
mean_mape_list_long = percentage_error_long.mean(dim=(1, 2))
mean_mape_long = percentage_error_long.mean()


# =====================
# MAE in t+5 (for the plot)
# =====================

h = 4  # t+5

abs_error_t5 = torch.abs(pred_3d_long[:, h, :] - target_3d_long[:, h, :])
mean_mae_t5_list = abs_error_t5.mean(dim=1)

print(f"Global RMSE: {mean_rmse_long:.4f}")
print(f"Global MAE: {mean_mae_long:.4f}")
print(f"Global MAPE: {mean_mape_long:.4f}")

In [ ]:
# Save the results to a CSV file
save_result_csv(
    "../results/lorenz_metrics.csv",
    {
        "forecasting": "long-term",
        "model": "iQTransformer",
        "MAPE": mean_mape_long,
        "MAE": mean_mae_long,
        "RMSE": mean_rmse_long,
    },
)

In [ ]:
pred_x_long = pred_3d_long[:, :, 0]
pred_y_long = pred_3d_long[:, :, 1]
pred_z_long = pred_3d_long[:, :, 2]

plot_preds_and_error(
    pred_x_long,
    pred_y_long,
    pred_z_long,
    mean_mae_t5_list,
    time_val,
    window_size,
    val_scaled,
    error="MAE",
    horizon=horizon_long,
    horizon_step=5,
    title="iQTransformer long-term t+5"
)